<a href="https://colab.research.google.com/github/Shahd799/flyrank-internship-1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahd799/flyrank-internship-1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Build model_df — this was missing from the notebook. Uses the same
# March/April construction as Week 6/7 so results stay consistent
# across the project.
import pandas as pd
import numpy as np

march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)
april = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet"
)

march_agg = (
    march.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_sum_position=("gsc_sum_position", "sum")
    )
)
march_agg["march_position"] = march_agg["march_sum_position"] / march_agg["march_impressions"]

april_agg = (
    april.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(april_impressions=("gsc_impressions", "sum"))
)

model_df = march_agg[march_agg["march_impressions"] > 0].copy()
model_df = model_df.merge(april_agg, on=["client_hash_id", "content_hash_id"], how="left")
model_df["april_impressions"] = model_df["april_impressions"].fillna(0)
model_df["target_decline"] = (model_df["april_impressions"] < model_df["march_impressions"]).astype(int)

model_df = model_df.dropna(subset=["march_impressions", "march_clicks", "march_position", "target_decline"])

print("Model rows:", len(model_df))
print(model_df["target_decline"].value_counts())

Model rows: 176738
target_decline
1    111968
0     64770
Name: count, dtype: int64


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I use a Decision Tree because this lane predicts whether a page's
impressions decline from March to April.

A Decision Tree is appropriate because it can capture simple non-linear
relationships between March search signals and the decline outcome, while
remaining interpretable.

The model uses only March features: impressions, clicks, and average
position. No April outcome information is used as a feature.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score

features = ["march_impressions", "march_clicks", "march_position"]
target = "target_decline"

print("Features:", features)
print("Target:", target)

Features: ['march_impressions', 'march_clicks', 'march_position']
Target: target_decline


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a grouped client-level split so that pages from the same client do
not appear in both training and test data.

This is more honest than a random row split because pages belonging to the
same client can share similar behavior.

The test set contains clients that were not present in the training set.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

X = model_df[features].copy()
y = model_df[target].copy()
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.22, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(model_df.iloc[train_idx]["client_hash_id"])
test_clients = set(model_df.iloc[test_idx]["client_hash_id"])

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Shared clients:", len(train_clients & test_clients))

Train rows: 133491
Test rows: 43247
Train clients: 36
Test clients: 11
Shared clients: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I train the Decision Tree on the training clients and evaluate it on the
held-out clients.

The baseline score is recomputed here in this notebook, using the exact
Week-4 rule formula (2 * log1p(march_impressions)), applied to the SAME
test rows and evaluated with the SAME metric (Average Precision) against
target_decline — not a hardcoded number from a previous run.

The comparison table also includes the base rate (share of positive class
in the test set), so the Average Precision numbers can be read relative to
what a skill-less ranking would achieve.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Model
model = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]
model_ap = average_precision_score(y_test, model_scores)

# Baseline — recomputed in this notebook, same formula as Week 4,
# same test rows, same metric
baseline_score_test = 2 * np.log1p(X_test["march_impressions"])
baseline_ap = average_precision_score(y_test, baseline_score_test)

# Base rate
base_rate = y_test.mean()

comparison = pd.DataFrame({
    "Method": ["Base rate (naive)", "Week-4 baseline (recomputed)", "Decision Tree"],
    "Average Precision": [base_rate, baseline_ap, model_ap]
})

print(comparison)

                         Method  Average Precision
0             Base rate (naive)           0.577566
1  Week-4 baseline (recomputed)           0.587307
2                 Decision Tree           0.611394


The Decision Tree's Average Precision is compared against both the
recomputed Week-4 baseline and the naive base rate, on the identical
held-out test set. This shows whether the model adds real skill beyond
the simple rule, not just beyond random guessing.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model makes both false-positive and false-negative errors, so the
predictions should be treated as decision-support rather than certainty.

The observed feature importance shows which March signal the tree leans on
most.

A recommendation can still be wrong because a page may decline for reasons
that are not represented by these three March signals — three concrete
wrong cases are shown below.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import confusion_matrix

predicted_labels = (model_scores >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, predicted_labels).ravel()
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)
print()

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)
print(feature_importance)

True negatives: 6947
False positives: 11322
False negatives: 7282
True positives: 17696

             feature  importance
0  march_impressions    0.420269
2     march_position    0.306840
1       march_clicks    0.272891


**Why these features plausibly relate to decline:**
- **march_impressions** (highest importance): pages with very high March
  visibility may be more likely to regress toward the mean in April — a
  well-known pattern in traffic data, not a claim about causation.
- **march_position**: a page's ranking position reflects how competitive
  its current visibility is, which plausibly relates to whether that
  visibility holds.
- **march_clicks**: engagement volume is a secondary signal correlated
  with impressions, contributing less independent information once
  impressions and position are already in the model.

In [6]:
# Three concrete wrong cases
results = X_test.copy()
results["actual"] = y_test.values
results["predicted_prob"] = model_scores
results["predicted_label"] = predicted_labels

wrong_cases = results[results["actual"] != results["predicted_label"]]

# Pick 3 diverse wrong examples: most confident false positive,
# most confident false negative, and a borderline case near 0.5
confident_fp = wrong_cases[
    (wrong_cases["actual"] == 0) & (wrong_cases["predicted_label"] == 1)
].sort_values("predicted_prob", ascending=False).head(1)

confident_fn = wrong_cases[
    (wrong_cases["actual"] == 1) & (wrong_cases["predicted_label"] == 0)
].sort_values("predicted_prob", ascending=True).head(1)

borderline = wrong_cases.iloc[
    (wrong_cases["predicted_prob"] - 0.5).abs().argsort()[:1]
]

three_wrong = pd.concat([confident_fp, confident_fn, borderline])
print(three_wrong[["march_impressions", "march_clicks", "march_position", "actual", "predicted_prob", "predicted_label"]])

        march_impressions  march_clicks  march_position  actual  \
149728                281             1        5.935943       0   
36165                4383            74        5.411362       1   
164388                  2             0        8.000000       1   

        predicted_prob  predicted_label  
149728        0.609703                1  
36165         0.317142                0  
164388        0.486912                0  


**Case 1 (predicted decline, prob = 0.61, actually did not decline):** the
model leaned toward "decline" here, but only moderately — 0.61 is close to
the 0.5 threshold, not a confident call. Because the tree has max_depth=3,
it only produces a small number of distinct probability values (as seen in
Week 7's granularity note), so even its strongest calls are not far from
the decision boundary. This page's March signals resembled declining
pages, but something outside the three features likely kept it stable.

**Case 2 (predicted no decline, prob = 0.32, actually declined):** similarly
moderate, not a confident call. High March visibility does not guarantee
April stability — factors like competitor changes or seasonality are
invisible to this feature set.

**Case 3 (predicted no decline, prob = 0.49, actually declined):** this is
the most genuinely uncertain case — the model's output sits almost exactly
at the decision boundary, which is an honest signal that this page needs
human judgment rather than an automated call either way.

Overall: because this shallow tree produces only a handful of distinct
probability values, none of its predictions should be read as strongly
confident — the model output is better used as a coarse priority tier than
a precise confidence score.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.